In [1]:
#Sheet Pvt Final
import pandas as pd

# === READ DATA ===
pvt_df = pd.read_excel("Pvt draft.xlsx")

# === OH AGGREGATION (BY PN AL78) ===
oh_df = (
    pvt_df
    .groupby("PN AL78", as_index=False)
    .agg({"OH": "mean"})
    .rename(columns={"OH": "Avrg.OH CM"})
)

# === MAIN AGGREGATION (BY PN NO SUFFIX / PN.Final) ===
main_df = (
    pvt_df
    .groupby("PN.Final", as_index=False)
    .agg({
        "PN used": "first",
        "Description": "first",

        "In Rcv Total": "mean",
        "Qty Resvered": "sum",
        "In Tr Total": "mean",
        "7 Months Prior": "sum",
        "OO CM": "sum",
        "OO NM": "sum",
        "OO N2M": "sum",

        "Curr. Month": "max",
        "Req CM": "max",
        "Req NM": "max",

        "PN AL78": "first"   # link key for OH merge
    })
)

# === MERGE OH INTO MAIN ===
result_df = (
    main_df
    .merge(oh_df, on="PN AL78", how="left")
)

# === RENAME COLUMNS ===
result_df = result_df.rename(columns={
    "PN.Final": "PN No suffix",
    "PN used": "Long PN",

    "In Rcv Total": "Avrg. InRcv",
    "Qty Resvered": "Sum Qty Resvrd",
    "In Tr Total": "Avrg. InTr",
    "7 Months Prior": "Sum 7 Months Prior",
    "OO CM": "Sum OO CM",
    "OO NM": "Sum OO NM",
    "OO N2M": "Sum OO N2M",
    "Curr. Month": "Max.Curr. Month",
    "Req CM": "Max.Req. CM",
    "Req NM": "Max.Req. NM",
})

# === FINAL COLUMN ORDER ===
result_df = result_df[[
    "PN AL78",
    "Long PN",
    "PN No suffix",
    "Description",
    "Avrg.OH CM",
    "Avrg. InRcv",
    "Sum Qty Resvrd",
    "Avrg. InTr",
    "Sum 7 Months Prior",
    "Sum OO CM",
    "Sum OO NM",
    "Sum OO N2M",
    "Max.Curr. Month",
    "Max.Req. CM",
    "Max.Req. NM",
]]

# === EXPORT ===
result_df.to_excel("Final_Summary.xlsx", index=False)


In [3]:
# === DEFINE COLUMN GROUPS ===
sum_cols = [
    "Sum Qty Resvrd",
    "Sum 7 Months Prior",
    "Sum OO CM",
    "Sum OO NM",
    "Sum OO N2M",
]

avg_cols = [
    "Avrg.OH CM",
    "Avrg. InRcv",
    "Avrg. InTr",
]

max_cols = [
    "Max.Curr. Month",
    "Max.Req. CM",
    "Max.Req. NM",
]

# === BUILD GRAND TOTAL ROW ===
grand_total = {}

for col in result_df.columns:
    if col in sum_cols:
        grand_total[col] = result_df[col].sum()
    elif col in avg_cols:
        grand_total[col] = result_df[col].mean()
    elif col in max_cols:
        grand_total[col] = result_df[col].max()
    else:
        grand_total[col] = "GRAND TOTAL"

# === APPEND GRAND TOTAL ROW ===
result_df = pd.concat(
    [result_df, pd.DataFrame([grand_total])],
    ignore_index=True
)


In [ ]:
# # === OPTIONAL: EXPORT ===
# result_df.to_excel("pvtfinal.xlsx", index=False)